# Lab 1: Inspecting Retrieval

**Workshop 1, block 3. 15 minutes.**

This lab does not improve anything. It gets you to look at what your
retriever actually returned, which most teams never do.

You leave with one number: how many of the 15 dev questions had their
answer present in the retrieved chunks. That is the ceiling on your
score, because if retrieval never found the fact, no prompt can recover
it.

> **Before you start:** `sample_chroma/` and `dev_set.json`, both in the workshop folder. You do not need a scraper yet.
>
> **When you finish:** Two numbers written down: retrieval success at k=5 and at k=10. Bring them to lab 2.

---
## Setup

Run these two cells first. They are identical in every lab, so each
notebook works on its own.

Your key is entered with `getpass`: not echoed, not written to disk, and
gone when the kernel stops. **Do not commit a notebook with a key
visible in its output.**

In [ ]:
# pip install openai chromadb beautifulsoup4 requests python-dotenv

import getpass, json, os, re, time, statistics
from pathlib import Path
from openai import AzureOpenAI

# Reads .env if you have one, otherwise uses the defaults below, so the
# notebook runs either way. Copy .env.example to .env to override.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview")
CHAT_BASE   = os.getenv("AZURE_CHAT_BASE",  "https://api-iw.azure-api.net/sig-shared-jpeast-increased")
EMBED_BASE  = os.getenv("AZURE_EMBED_BASE", "https://api-iw.azure-api.net/sig-embedding")

CHAT_DEPLOYMENT   = os.getenv("CHAT_DEPLOYMENT",   "gpt-4o-mini")
VISION_DEPLOYMENT = os.getenv("VISION_DEPLOYMENT", "gpt-5-mini")
EMBED_DEPLOYMENT  = os.getenv("EMBED_DEPLOYMENT",  "text-embedding-3-small")

# This gateway takes the FULL path as the endpoint: deployment, operation
# and api-version included. So a client is bound to one deployment, and
# we need three of them.
#
# The chat route has no /openai segment, the embedding route does. That
# asymmetry is real, so do not tidy them into matching.

CHAT_URL   = f"{CHAT_BASE}/deployments/{CHAT_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
VISION_URL = f"{CHAT_BASE}/deployments/{VISION_DEPLOYMENT}/chat/completions?api-version={API_VERSION}"
EMBED_URL  = f"{EMBED_BASE}/openai/deployments/{EMBED_DEPLOYMENT}/embeddings?api-version={API_VERSION}"

# .strip() matters: pasting into a prompt often picks up a trailing
# newline, and that alone produces a 401.
KEY = (os.getenv("AZURE_OPENAI_KEY")
       or getpass.getpass("Azure OpenAI key: ")).strip()


def _client(url):
    return AzureOpenAI(azure_endpoint=url, api_key=KEY, api_version=API_VERSION)


chat_client   = _client(CHAT_URL)
vision_client = _client(VISION_URL)
embed_client  = _client(EMBED_URL)

# Each is tested separately so one failure does not hide the others.
for label, fn in [
    ("chat  ", lambda: chat_client.chat.completions.create(
        model=CHAT_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("vision", lambda: vision_client.chat.completions.create(
        model=VISION_DEPLOYMENT, max_tokens=5,
        messages=[{"role": "user", "content": "Reply with one word: connected"}]
     ).choices[0].message.content.strip()),
    ("embed ", lambda: f"{len(embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=['test']).data[0].embedding)} dimensions"),
]:
    try:
        print(f"{label}  OK      {fn()}")
    except Exception as e:
        print(f"{label}  FAILED  {type(e).__name__}: {str(e)[:110]}")

# 401 means the path is right and the key is wrong.
# 404 means the path is wrong, not the key.

In [ ]:
# ---- helpers ------------------------------------------------------

def chat(messages, model=None, temperature=0.0, max_tokens=512):
    r = chat_client.chat.completions.create(
        model=model or CHAT_DEPLOYMENT, messages=messages,
        temperature=temperature, max_tokens=max_tokens)
    return r.choices[0].message.content or ""


def ask(prompt, system=None, **kw):
    msgs = ([{"role": "system", "content": system}] if system else [])
    return chat(msgs + [{"role": "user", "content": prompt}], **kw)


def embed(texts, batch_size=256):
    """Embed a LIST of strings. One call per string is the slow mistake:
    3,000 round trips at ~200ms each is ten minutes of network wait."""
    texts = [t.replace("\n", " ") for t in texts]
    out = []
    for i in range(0, len(texts), batch_size):
        r = embed_client.embeddings.create(model=EMBED_DEPLOYMENT, input=texts[i:i + batch_size])
        out.extend(d.embedding for d in r.data)
    return out


def chunk(text, size=800, overlap=100):
    """Fixed-size chunks with overlap. Defaults to start from, not
    recommended values."""
    text = " ".join(text.split())
    step = size - overlap
    return [text[i:i + size] for i in range(0, len(text), step)
            if text[i:i + size].strip()]


# One PersistentClient per path, cached for the life of this kernel.
# chromadb caches internal state per path, so deleting the folder and
# opening a fresh PersistentClient while an earlier one from this same
# session is still alive corrupts the connection: you get "attempt to
# write a readonly database" or "database is locked" on the very next
# call. Rebuilding your index more than once per session, which the
# "change one setting, re-run" loop asks you to do, hits this every
# time with the naive version.
_stores = {}


def get_store(path="data/chroma", name="workshop", reset=False):
    import chromadb, shutil
    from pathlib import Path as _P

    if path not in _stores:
        # First time this path is opened in this session. Safe to wipe a
        # stale, wrong-chromadb-version index here, since no client for
        # this path exists in this process yet.
        if reset and _P(path).exists():
            shutil.rmtree(path)
        try:
            _stores[path] = chromadb.PersistentClient(path=path)
        except KeyError as e:
            raise RuntimeError(
                f"chromadb cannot read the index at {path} ({e}). It was built by "
                f"a different chromadb version. Delete that folder and rebuild, or "
                f"install the pinned version from requirements.txt."
            ) from None

    client = _stores[path]
    if reset:
        # Reset now means delete-and-recreate the COLLECTION on the same
        # client, not delete-and-recreate the DIRECTORY under it. This is
        # what actually avoids the readonly/locked error on every rebuild
        # after the first.
        try:
            client.delete_collection(name)
        except Exception:
            pass
    return client.get_or_create_collection(name)


def add_to_store(store, texts, metadatas, ids=None, batch_size=128):
    ids = ids or [f"c{i}" for i in range(len(texts))]
    for i in range(0, len(texts), batch_size):
        sl = slice(i, i + batch_size)
        store.add(ids=ids[sl], documents=texts[sl],
                  embeddings=embed(texts[sl]), metadatas=metadatas[sl])


def query(store, question, k=5, where=None):
    """The k nearest chunks. Chroma returns squared L2, so lower is closer."""
    r = store.query(query_embeddings=embed([question]), n_results=k,
                    where=where or None)
    return [{"text": d, "metadata": m, "distance": dist}
            for d, m, dist in zip(r["documents"][0], r["metadatas"][0],
                                  r["distances"][0])]


def show(chunks, chars=200):
    if not chunks:
        print("  (nothing returned)")
        return
    for c in chunks:
        print(f"  {c['distance']:.3f}  {c['metadata'].get('url', '?')}")
        print(f"         {c['text'][:chars].strip()}\n")


def timed(fn, *a, **kw):
    t0 = time.time()
    return fn(*a, **kw), time.time() - t0


def normalise(s):
    return re.sub(r"[^a-z0-9 ]", " ", (s or "").lower())


def answer_present(expected, chunks):
    """Was the expected answer anywhere in the retrieved text? Crude, and
    enough to tell a retrieval failure from a prompt failure."""
    hay  = normalise(" ".join(c["text"] for c in chunks))
    need = normalise(expected).strip()
    if need and need in hay:
        return True
    terms = [t for t in need.split() if len(t) > 3]
    return bool(terms) and sum(t in hay for t in terms) / len(terms) >= 0.8


def load_dev_set(path="dev_set.json"):
    return json.loads(Path(path).read_text())


def describe_image(image_path, prompt, model=None):
    """One image plus an instruction. There is deliberately no default
    prompt: writing it is the lab 4 exercise."""
    import base64, mimetypes
    p    = Path(image_path)
    mime = mimetypes.guess_type(p.name)[0] or "image/jpeg"
    b64  = base64.b64encode(p.read_bytes()).decode()
    r = vision_client.chat.completions.create(
        model=model or VISION_DEPLOYMENT, max_tokens=800,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        ]}])
    return r.choices[0].message.content or ""

print("helpers loaded")

---
## Step 1: Three questions

Pick three: one the corpus answers plainly, one it answers only
indirectly, one about something not in the corpus at all.

Question 3 matters most. Watch how many results come back and how the
distances compare with question 1.

In [ ]:
store = get_store("sample_chroma", name="sample")
dev   = load_dev_set("dev_set.json")
print(f"{store.count()} chunks, {len(dev)} dev questions")

QUESTIONS = [
    "What is the capacity of Makerspace A?",       # answered plainly
    "Where would I go to use a laser cutter?",     # answered indirectly
    "What is the wifi password in the library?",   # not in the corpus
]

for q in QUESTIONS:
    print("=" * 70)
    print("Q:", q)
    print("=" * 70)
    show(query(store, q, k=5))

### Step 2: Mark each one

Was the expected answer present anywhere in the five chunks? Yes or no.
That is the only thing you record.

Notice question 3 still returned five results. Vector search never
reports "no match": ask for 5 and you get 5, however unrelated. This is
why the scoring strategy is to always answer.

---
## Step 3: All 15 dev questions

In [ ]:
def retrieval_rate(store, dev, k=5, verbose=True):
    hits = 0
    for item in dev:
        chunks = query(store, item["question"], k=k)
        ok = answer_present(item["answer"], chunks)
        hits += ok
        if verbose:
            print(f"  {'yes' if ok else 'NO '}  Lv{item['level']}  {item['question'][:58]}")
    print(f"\nRetrieval success at k={k}: {hits}/{len(dev)}")
    return hits


hits_k5 = retrieval_rate(store, dev, k=5)

---
## Step 4: Change k and repeat

More chunks raise the chance of including the answer, and the chance of
adding noise.

In [ ]:
hits_k10 = retrieval_rate(store, dev, k=10, verbose=False)

print(f"\nk=5   {hits_k5}/{len(dev)}")
print(f"k=10  {hits_k10}/{len(dev)}")
print(f"Gained {hits_k10 - hits_k5} questions for double the context.")

### If you finish early

Sweep k properly. The count usually rises then flattens, and the flat
point is where extra chunks only add noise and cost.

In [ ]:
for k in (3, 5, 8, 10, 15):
    print(f"k={k:<3} ", end="")
    retrieval_rate(store, dev, k=k, verbose=False)

---
## What to take away

| What you found | What it means | What to change |
|---|---|---|
| Answer not in the chunks | Retrieval problem | Chunking, k, or search method |
| Answer in the chunks, reply still wrong | Prompt problem | The prompt, and only then |

Print the chunks before changing anything.

**Write your two numbers down.** Lab 2 compares against them.